<a href="https://colab.research.google.com/github/yazflores/MIAAD/blob/main/Yazmin_Flores_Transformaciones_con_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#         UNIVERSIDAD AUTONOMA DE CIUDAD JUAREZ (UACJ)          
##    Maestría en Inteligencia Artificial y Analítica de Datos    

  INGENIERÍA DE DATOS AVANZADA -  con PySpark
                                                                
  Alumna: Yazmin Ivonne Flores Martinez - Matrícula 261548.     
  Profesor: Dr. Vicente García Jiménez                   

  Link Colab: https://colab.research.google.com/drive/1mOGLZvgLIVBvjZIFrCmO6dJXJ9uwekje#scrollTo=spE9ZNktzvnW

In [1]:
#@title 0 — Instalación y configuración del entorno

#============================================================
# VERIFICAR SI PYSPARK ESTÁ INSTALADO
# ============================================================

try:
    import pyspark

    print("PySpark ya está instalado.")
    print(f"Versión instalada: {pyspark.__version__}")

except ImportError:

    print("PySpark no está instalado.")
    print("Instalando PySpark 3.5.0...\n")

    # Instalación silenciosa
    !pip install pyspark==3.5.0 -q

    # Verificación posterior a la instalación
    import pyspark

    print("PySpark instalado correctamente.")
    print(f"Versión instalada: {pyspark.__version__}")

# ============================================================
# MOSTRAR VERSIÓN DESDE TERMINAL
# ============================================================

print("\nInformación desde terminal:")
!pyspark --version

PySpark ya está instalado.
Versión instalada: 4.0.2

Información desde terminal:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.0.2
      /_/
                        
Using Scala version 2.13.16, OpenJDK 64-Bit Server VM, 17.0.18
Branch HEAD
Compiled by user runner on 2026-02-02T08:08:13Z
Revision 7cc3b9bcdaab8c923f23cdbc9ce922530e1becf1
Url https://github.com/apache/spark
Type --help for more information.


In [8]:
#@title 0.5 - Inicialización del Entorno de Trabajo en PySpark

# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================

# Librería estándar de Python para interactuar con el sistema operativo
import os

# Clase principal para crear una sesión de Spark
from pyspark.sql import SparkSession

# Módulo de funciones SQL de PySpark
# Se importa como F para facilitar su uso
from pyspark.sql import functions as F

# Tipos de datos utilizados para definir esquemas
from pyspark.sql.types import (
    StructType,      # Permite definir la estructura completa del DataFrame
    StructField,     # Permite definir cada columna del esquema
    StringType,      # Tipo cadena de texto
    IntegerType,     # Tipo entero
    DoubleType,      # Tipo numérico decimal
    TimestampType    # Tipo fecha y hora
)

# Librerías para visualización en Google Colab/Jupyter
from IPython.display import display, HTML

# Librería para manipulación de datos tabulares
import pandas as pd


# ============================================================
# CREACIÓN DE LA SPARK SESSION
# ============================================================

# SparkSession es el punto de entrada principal para trabajar con PySpark.
# A través de ella se pueden crear DataFrames, ejecutar consultas SQL
# y acceder a las funcionalidades del motor Spark.

spark = (
    SparkSession.builder

    # Nombre de la aplicación que aparecerá en la interfaz de Spark
    .appName('TransformacionesAdicionales_NSLKDD')

    # Desactiva la barra de progreso en consola
    # para obtener una salida más limpia en Colab
    .config('spark.ui.showConsoleProgress', 'false')

    # Crea la sesión o recupera una existente
    .getOrCreate()
)

# Reduce la cantidad de mensajes del log
# Mostrará únicamente errores importantes
spark.sparkContext.setLogLevel('ERROR')


# ============================================================
# VERIFICAR LA SESIÓN DE SPARK
# ============================================================

# Imprime la versión de Spark activa
print(f'✅ SparkSession activa — versión Spark: {spark.version}')

✅ SparkSession activa — versión Spark: 4.0.2


In [17]:
#@title Importa la librería de Google Colab para Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [49]:
#@title 1 — Carga del dataset y exploración inicial

# ============================================================
# Definición del Esquema
# ============================================================

from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, StringType

schema = StructType([
    StructField("duration", StringType(), True),
    StructField("protocol_type", StringType(), True),
    StructField("service", StringType(), True),
    StructField("flag", IntegerType(), True),
    StructField("src_bytes", IntegerType(), True),
    StructField("dst_bytes", IntegerType(), True),
    StructField("land", IntegerType(), True),
    StructField("wrong_fragment", IntegerType(), True),
    StructField("urgent", IntegerType(), True),
    StructField("hot", IntegerType(), True),
    StructField("num_failed_logins", IntegerType(), True),
    StructField("logged_in", IntegerType(), True),
    StructField("num_compromised", IntegerType(), True),
    StructField("root_shell", IntegerType(), True),
    StructField("su_attempted", IntegerType(), True),
    StructField("num_root", IntegerType(), True),
    StructField("num_file_creations", IntegerType(), True),
    StructField("num_shells", IntegerType(), True),
    StructField("num_access_files", IntegerType(), True),
    StructField("num_outbound_cmds", IntegerType(), True),
    StructField("is_host_login", IntegerType(), True),
    StructField("is_guest_login", IntegerType(), True),
    StructField("count", IntegerType(), True),
    StructField("srv_count", FloatType(), True),
    StructField("serror_rate", FloatType(), True),
    StructField("srv_serror_rate", FloatType(), True),
    StructField("rerror_rate", FloatType(), True),
    StructField("srv_rerror_rate", FloatType(), True),
    StructField("same_srv_rate", FloatType(), True),
    StructField("diff_srv_rate", FloatType(), True),
    StructField("srv_diff_host_rate", IntegerType(), True),
    StructField("dst_host_count", IntegerType(), True),
    StructField("dst_host_srv_count", FloatType(), True),
    StructField("dst_host_same_srv_rate", FloatType(), True),
    StructField("dst_host_diff_srv_rate", FloatType(), True),
    StructField("dst_host_same_src_port_rate", FloatType(), True),
    StructField("dst_host_srv_diff_host_rate", FloatType(), True),
    StructField("dst_host_serror_rate", FloatType(), True),
    StructField("dst_host_srv_serror_rate", FloatType(), True),
    StructField("dst_host_rerror_rate", FloatType(), True),
    StructField("dst_host_srv_rerror_rate", StringType(), True),  # objeto, revisar conversión
    StructField("class", IntegerType(), True)
])

# ============================================================
# CARGA DEL CONJUNTO DE DATOS CSV
# ============================================================

# Ruta del archivo CSV almacenado en Google Drive
ruta_csv = '/content/drive/MyDrive/PySparkData/NSL_KDD.csv'

# Cargar el archivo CSV en un DataFrame de PySpark
df_raw = (
    spark.read

    # Indica que el archivo contiene encabezados
    .option('header', 'true')

    # Aplicar esquema manual
    .schema(schema)

    # Cargar archivo CSV
    .csv(ruta_csv)
)

# ============================================================
# Renombrar columnas 'class' a 'label' por consistencia
# Ajustar el desfase de las columnas
# ============================================================

# Definir la lista correcta de nombres de columnas
columnas_correctas = [
    'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label','duration'
]

# Reasignar los nombres de columnas al DataFrame
df_raw = df_raw.toDF(*columnas_correctas)

# ============================================================
# MOSTRAR INFORMACIÓN GENERAL DEL DATAFRAME
# ============================================================

print('Esquema:\n')
df_raw.printSchema()


Esquema:

root
 |-- protocol_type: string (nullable = true)
 |-- service: string (nullable = true)
 |-- flag: string (nullable = true)
 |-- src_bytes: integer (nullable = true)
 |-- dst_bytes: integer (nullable = true)
 |-- land: integer (nullable = true)
 |-- wrong_fragment: integer (nullable = true)
 |-- urgent: integer (nullable = true)
 |-- hot: integer (nullable = true)
 |-- num_failed_logins: integer (nullable = true)
 |-- logged_in: integer (nullable = true)
 |-- num_compromised: integer (nullable = true)
 |-- root_shell: integer (nullable = true)
 |-- su_attempted: integer (nullable = true)
 |-- num_root: integer (nullable = true)
 |-- num_file_creations: integer (nullable = true)
 |-- num_shells: integer (nullable = true)
 |-- num_access_files: integer (nullable = true)
 |-- num_outbound_cmds: integer (nullable = true)
 |-- is_host_login: integer (nullable = true)
 |-- is_guest_login: integer (nullable = true)
 |-- count: integer (nullable = true)
 |-- srv_count: integer (null

Conclusión: Este dataset es una simulación de tráfico de red con ataques y tráfico normal, diseñado para estudiar patrones de comportamiento y entrenar sistemas de detección de intrusiones. Permite analizar:

Protocolos más usados
Servicios más atacados
Diferencias de volumen entre ataques y tráfico normal
Outliers y picos de tráfico que indican intrusiones críticas

In [50]:
#@title 2. Se muesta el total de filas, columnas y las primeras 10 filas usando comando SHOW

print("\nNúmero total de filas:", df_raw.count())
print("\nNúmero total de columnas:", len(df_raw.columns))
df_raw.show(10)


Número total de filas: 18035

Número total de columnas: 42
+-------------+-------+----+---------+---------+----+--------------+------+---+-----------------+---------+---------------+----------+------------+--------+------------------+----------+----------------+-----------------+-------------+--------------+-----+---------+-----------+---------------+-----------+---------------+-------------+-------------+------------------+--------------+------------------+----------------------+----------------------+---------------------------+---------------------------+--------------------+------------------------+--------------------+------------------------+------------+--------+
|protocol_type|service|flag|src_bytes|dst_bytes|land|wrong_fragment|urgent|hot|num_failed_logins|logged_in|num_compromised|root_shell|su_attempted|num_root|num_file_creations|num_shells|num_access_files|num_outbound_cmds|is_host_login|is_guest_login|count|srv_count|serror_rate|srv_serror_rate|rerror_rate|srv_rerror_rat

Conclusión: Es un dataset completo y balanceado para análisis de seguridad en redes, con suficiente tamaño y complejidad para sacar conclusiones confiables sobre protocolos, servicios y patrones de ataques.

In [51]:
#@title 3. Seleccionar columnas usando SELECT()
#Seleccionar únicamente las siguientes columnas:
# protocol_type
# service
# src_bytes
# dst_bytes
# label

df_raw_select = df_raw.select('protocol_type',
                              'service',
                              'src_bytes',
                              'dst_bytes',
                              'label')
df_raw_select.show(10)

+-------------+-------+---------+---------+------------+
|protocol_type|service|src_bytes|dst_bytes|       label|
+-------------+-------+---------+---------+------------+
|          tcp|private|        0|        0|     neptune|
|          tcp|private|        0|        0|     neptune|
|          tcp| telnet|        0|       15|       mscan|
|          tcp|   http|      267|    14515|      normal|
|          tcp| telnet|      129|      174|guess_passwd|
|          tcp|   http|      327|      467|      normal|
|          tcp|    ftp|       26|      157|guess_passwd|
|          tcp| telnet|        0|        0|       mscan|
|          tcp|private|        0|        0|     neptune|
|          tcp| telnet|        0|        0|     neptune|
+-------------+-------+---------+---------+------------+
only showing top 10 rows


Conclusión: El select permite enfocarse en las columnas clave para análisis de intrusiones: protocolo, servicio, tráfico y etiqueta.
Las primeras 10 filas muestran claramente una combinación de tráfico nulo (escaneos/ataques) y tráfico activo (normal o ataques con payload).


In [54]:
#@title 4. Filtrar registros usando FILTER()
#Filtrar únicamente los registros donde:
#protocol_type = 'tcp'
#y src_bytes > 1000

from pyspark.sql import functions as F

df_filtrado = df_raw.filter(
    (F.col('protocol_type') == 'tcp') & (F.col('src_bytes') > 1000)
)

Total_filas = df_filtrado.count()


print(f' Total de Registros que cumplen las condiciones: {Total_filas:,}\n')
# Mostrar los primeros 10 registros filtrados
df_filtrado.show(10, truncate=False)

 Total de Registros que cumplen las condiciones: 1,552

+-------------+--------+----+---------+---------+----+--------------+------+---+-----------------+---------+---------------+----------+------------+--------+------------------+----------+----------------+-----------------+-------------+--------------+-----+---------+-----------+---------------+-----------+---------------+-------------+-------------+------------------+--------------+------------------+----------------------+----------------------+---------------------------+---------------------------+--------------------+------------------------+--------------------+------------------------+-----------+--------+
|protocol_type|service |flag|src_bytes|dst_bytes|land|wrong_fragment|urgent|hot|num_failed_logins|logged_in|num_compromised|root_shell|su_attempted|num_root|num_file_creations|num_shells|num_access_files|num_outbound_cmds|is_host_login|is_guest_login|count|srv_count|serror_rate|srv_serror_rate|rerror_rate|srv_rerror_rate|s

Conclusión: El filtrado permitió aislar registros TCP con actividad significativa (src_bytes > 1000).
Se destacan ataques volumétricos y servicios http, ftp_data como los más relevantes para análisis de seguridad.


In [56]:
#@title 5. Crear una nueva columna llamada: total_bytes con WITHCOLUMN()
#La nueva columna deberá contener:
#src_bytes + dst_bytes

# Crear nueva columna total_bytes sumando src_bytes + dst_bytes
df_raw = df_raw.withColumn('total_bytes', F.col('src_bytes') + F.col('dst_bytes'))

# Mostrar las columnas implicadas y las primeras 10 filas para verificar
df_raw.select('protocol_type', 'service', 'src_bytes', 'dst_bytes', 'total_bytes').show(10, truncate=False)


+-------------+-------+---------+---------+-----------+
|protocol_type|service|src_bytes|dst_bytes|total_bytes|
+-------------+-------+---------+---------+-----------+
|tcp          |private|0        |0        |0          |
|tcp          |private|0        |0        |0          |
|tcp          |telnet |0        |15       |15         |
|tcp          |http   |267      |14515    |14782      |
|tcp          |telnet |129      |174      |303        |
|tcp          |http   |327      |467      |794        |
|tcp          |ftp    |26       |157      |183        |
|tcp          |telnet |0        |0        |0          |
|tcp          |private|0        |0        |0          |
|tcp          |telnet |0        |0        |0          |
+-------------+-------+---------+---------+-----------+
only showing top 10 rows


Conclusión: La mayor parte del tráfico activo en estas filas se concentra en http.Muchos registros son inactivos (total_bytes=0), mostrando que no todo TCP genera tráfico útil.
La columna total_bytes permite identificar rápidamente sesiones o ataques volumétricos frente a intentos nulos o exploratorios.

In [58]:
#@title 6. Ordenar registros de mayor a menor utilizando ORDERBY()
# Ordenar los registros de mayor a menor utilizando la columna:total_bytes

df_ordenado = df_raw.orderBy(F.col('total_bytes').desc())

# Mostrar las primeras 10 filas del DataFrame ordenado
df_ordenado.select('protocol_type', 'service', 'src_bytes', 'dst_bytes', 'total_bytes').show(10, truncate=False)


+-------------+--------+---------+---------+-----------+
|protocol_type|service |src_bytes|dst_bytes|total_bytes|
+-------------+--------+---------+---------+-----------+
|tcp          |X11     |62825648 |90476    |62916124   |
|tcp          |X11     |31645608 |207796   |31853404   |
|tcp          |ftp_data|6291668  |0        |6291668    |
|tcp          |ftp_data|3131464  |0        |3131464    |
|tcp          |ftp_data|2194619  |0        |2194619    |
|tcp          |telnet  |347      |1345927  |1346274    |
|tcp          |telnet  |270      |1288652  |1288922    |
|tcp          |telnet  |115      |1285078  |1285193    |
|tcp          |X11     |13948    |1171108  |1185056    |
|tcp          |telnet  |256      |925244   |925500     |
+-------------+--------+---------+---------+-----------+
only showing top 10 rows


Conclusión: Los registros con mayor total_bytes corresponden a servicios X11 y ftp_data, lo que indica ataques o transferencias de alto volumen.
Las técnicas de detección de anomalías pueden aprovechar este ordenamiento para identificar fácilmente los ataques volumétricos.
El patrón TCP con alto src_bytes y dst_bytes sirve como indicador de tráfico sospechoso en análisis de seguridad.

In [59]:
#@title 7. Mostrar los valores únicos de: protocol_type usando DISTINCT()

df_raw.select('protocol_type').distinct().show(truncate=False)

+-------------+
|protocol_type|
+-------------+
|tcp          |
|udp          |
|icmp         |
+-------------+



Conclusión: La tabla te muestra que el dataset trabaja solo con tres tipos de protocolos de red, y esto guía cómo segmentar y analizar el tráfico de ataques y normal.

In [60]:
#@title 8. Agrupar los registros usando GROUPBY()
#Agrupar los registros utilizando:
#  groupBy('protocol_type')
#  y calcular: el número de registros utilizando count().


df_raw.groupBy('protocol_type').count().show(truncate=False)

+-------------+-----+
|protocol_type|count|
+-------------+-----+
|tcp          |15136|
|udp          |2074 |
|icmp         |825  |
+-------------+-----+



Conclusión: El dataset está fuertemente sesgado hacia TCP, por lo que los análisis de bytes y ataques volumétricos estarán dominados por TCP.
UDP e ICMP representan un pequeño porcentaje de registros, pero no deben ignorarse, porque aunque sean pocos, suelen corresponder a ataques específicos (DoS, scans, exploits).
Para construir reglas de detección o modelos de machine learning, es importante considerar el desbalance entre protocolos.

In [66]:
#@title Agrupar usando GROUPBY y AVG para calcular el promedio
# Agrupar por label y calcular: el promedio de src_bytes.

# Agrupar por 'label', calcular promedio y redondear a 4 decimales
df_promedio = df_raw.groupBy('label') \
    .agg(F.round(F.avg('src_bytes'), 4).alias('avg_src_bytes'))

# Ordenar de mayor a menor por el promedio AVG
df_ordenado = df_promedio.orderBy(F.col('avg_src_bytes').desc())

# Mostrar todos los registros completos
df_ordenado.show(df_ordenado.count(), truncate=False)

+---------------+---------------+
|label          |avg_src_bytes  |
+---------------+---------------+
|xlock          |1.35040994286E7|
|warezmaster    |65186.2585     |
|rootkit        |54713.2        |
|back           |52894.3505     |
|apache2        |38689.7547     |
|named          |22001.6429     |
|worm           |4209.0         |
|xterm          |3137.9091      |
|mailbomb       |2599.0         |
|normal         |2501.9257      |
|buffer_overflow|1867.0         |
|multihop       |1767.8333      |
|sendmail       |1698.7273      |
|imap           |1352.0         |
|pod            |1341.2121      |
|smurf          |909.5143       |
|xsnoop         |775.75         |
|httptunnel     |506.8365       |
|sqlattack      |398.0          |
|loadmodule     |277.0          |
|perl           |258.0          |
|ps             |124.5833       |
|snmpgetattack  |106.5263       |
|guess_passwd   |57.5884        |
|snmpguess      |45.4562        |
|ftp_write      |37.0           |
|teardrop     

Conclusión: El promedio de src_bytes permite diferenciar ataques volumétricos de tráfico normal y ataques discretos.
Los outliers (xlock, warezmaster) son fácilmente detectables mediante reglas simples de umbral de bytes.
Los ataques de bajo o nulo volumen requieren análisis multivariado: correlación con dst_bytes, flag, service y patrones temporales.
Esta tabla es útil para priorizar recursos en detección de intrusiones, enfocándose primero en ataques volumétricos y luego en los stealth.